# V3-4B — A-MIL: Gated Attention over video bags

هر نمونه یک MP4 است، نه یک فریم یا یک کلیپ مستقل. مدل فقط featureهای RGB و برچسب سطح video را می‌بیند؛ زمان رخداد فقط هنگام ساخت bag آموزشی استفاده و در manifest ثبت می‌شود.

In [ ]:
from __future__ import annotations
from pathlib import Path
import sys
from IPython.display import display
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT / 'scripts') not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / 'scripts'))

from v3_attention_mil import Config, create_selected_bags, context_report, cache_preflight, ensure_feature_cache, train_model, evaluate_best

config = Config()
# قبل از اجرای چندساعته، preflight را ببین. اجرای کامل فقط با True آغاز می‌شود.
RUN_FULL_FEATURE_CACHE = False
MAX_NEW_SEQUENCES = None
RUN_TRAINING_AFTER_COMPLETE_CACHE = True
print({'data_root': str(config.data_root), 'max_train_windows': config.max_train_windows, 'attention_dim': config.attention_dim, 'run_full_feature_cache': RUN_FULL_FEATURE_CACHE})

In [ ]:
# 1) Create one documented bag per video: train bags are capped at 8; validation bags retain all sliding windows.
context = create_selected_bags(config)
print(context_report(context))
display(context['selected_bags'].groupby(['split', 'video_label']).agg(bags=('bag_id', 'size'), selected_windows=('selected_window_count', 'sum'), mean_core=('selected_positive_core_count', 'mean'), mean_hard_negative=('selected_hard_negative_count', 'mean')))
assert context['selected_bags'].video_id.nunique() == 600


In [ ]:
# 2) Two-window preflight using the exact frozen ResNet18 preprocessing.
preflight = cache_preflight(context)
print(preflight)


In [ ]:
# 3) Resumable feature cache. Existing V3-3 and V3-4A features are reused first.
cache_result = ensure_feature_cache(context, run_full_cache=RUN_FULL_FEATURE_CACHE, max_sequences=MAX_NEW_SEQUENCES)
print({key: value for key, value in cache_result.items() if key not in {'features_by_sequence', 'feature_source_by_sequence'}})
display(pd.Series(cache_result['feature_source_by_sequence']).value_counts().rename_axis('feature_source').to_frame('sequences'))
if not cache_result['complete']:
    print('Cache is incomplete. Set RUN_FULL_FEATURE_CACHE=True only when no other extraction is running.')


In [ ]:
# 4) Train gated attention MIL with video-level labels only.
training_result = None
if cache_result['complete'] and RUN_TRAINING_AFTER_COMPLETE_CACHE:
    training_result = train_model(context, cache_result)
    print(training_result)
else:
    print('Training waits for the complete A-MIL cache.')


In [ ]:
# 5) Full-MP4 validation and attention provenance.
evaluation_result = None
if cache_result['complete'] and config.best_model_path.is_file():
    evaluation_result = evaluate_best(context, cache_result)
    print(evaluation_result['summary']['metrics_validation_selected_threshold'])
    display(evaluation_result['videos'].sort_values('video_probability', ascending=False).head(12))
    display(evaluation_result['attention'].sort_values(['video_id', 'attention_rank_in_video']).head(20))
else:
    print('Evaluation waits for the trained checkpoint.')
